# 01. Shared service credential

The agent calls the tool with a single static API key on every request,
for every user. The tool has no idea who is asking.

This is where most early agentic prototypes start. The agent is its own
trusted system, and "auth" between agent and tool is a shared secret that
proves "I am the agent". The tool returns whatever the agent asks for, and
the agent is responsible for showing the user the right subset of what came
back.

Notebooks 2 through 8 each fix one thing that's broken about this baseline.

## What this pattern looks like on the wire

```
   alice -+
          |
   bob   -+--->  agent  -- X-API-Key: dev-shared-api-key -->  expense-service
          |
   dave  -+
```

The arrows from the users to the agent might be HTTP, a chat UI, a Slack
bot. That part isn't the point. The point is the right-hand arrow: exactly
the same header on every request, regardless of which user prompted the agent.

In [ ]:
from shared.agent import Agent
from shared.auth import ServiceCredentialAuth
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw
from shared.config import EXPENSE_SERVICE_URL

strategy = ServiceCredentialAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## Three users, same prompt

Watch the answers carefully. The agent has no information about who is
asking. Whatever filtering happens is done by the LLM reading the data, not
by the system.

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)

In [ ]:
result_dave = run_as("dave", "Show me all the expenses across the company.", agent)

## What did the tool actually see?

The agent worked hard to format the right answer for each user. Here is
what the service actually got on its most recent request.

In [ ]:
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Tradeoffs

- **Authz lives nowhere.** Or rather, in the LLM's prompt-following ability,
  which is not a security boundary.
- **The tool sees no real user.** Method on the service side is `api_key`.
  The service has no idea whether alice or bob or dave asked.
- **No cryptographic proof of identity.** Anyone with the API key is the agent.
- **No least privilege.** The API key is fully privileged.
- **Audit trail:** "agent did X" is the most you can say. You cannot answer
  "did alice access bob's expenses?" from the service logs alone.
- **Prompt injection blast radius:** an attacker who gets the agent to do
  the wrong thing has full read access to every user's data, because the
  tool has no scoping.

What does work: the agent is simple, and the service is simple. Nothing
depends on a complex identity infrastructure. For a single-tenant internal
tool with no privacy or compliance constraints, this might genuinely be enough.

## What's still missing

The agent received three different prompts from three different users, and
in each case the LLM tried its best to filter the result *in its answer*.
The service returned the same data regardless of who asked. We're trusting
the LLM not to leak information across users, and the LLM is not a security
boundary.

The most direct fix is to push some awareness of "who is asking" from the
user, through the agent, into the tool call itself. Different patterns do
this differently, and each has its own failure modes.

Notebook 02 takes the smallest possible step: the agent reads the user's
identity claims and uses them to construct narrower tool calls. The tool
still has no idea who's asking, but at least the call is scoped before it
goes out.